# PesaPap Growth Analytics

## Project Objective

This project analyses PesaPap's user growth, transaction performance, customer behaviour and onboarding experiment. The goal is to use SQL, business KPIs, RFM segmentation, cohort retention and A/B testing to help the Head of Growth understand performance and decide whether to roll out the simplified onboarding screen.

## Database Tables

- `users`: one row per user. Primary key: `user_id`.
- `transactions`: one row per transaction. Primary key: `txn_id`. Foreign key: `user_id` references `users.user_id`.
- `experiment`: one row per experiment participant. Primary key: `user_id`. Foreign key: `user_id` references `users.user_id`.

In [1]:
%%writefile generate_data.py
import numpy as np, pandas as pd

rng = np.random.default_rng(77)

users = pd.DataFrame({
    "user_id": range(1, 3001),

    "signup_date": (pd.to_datetime("2026-01-01")
                    + pd.to_timedelta(rng.integers(0, 120, 3000), unit="D"))
                   .strftime("%Y-%m-%d"),

    "county": rng.choice(
        ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"],
        3000
    ),

    "acquisition_channel": rng.choice(
        ["Referral", "Social", "Agent", "Search"],
        3000
    ),
})

m = 12000

txns = pd.DataFrame({
    "txn_id": range(1, m + 1),

    "user_id": rng.integers(1, 3001, m),

    "txn_date": (pd.to_datetime("2026-01-15")
                 + pd.to_timedelta(rng.integers(0, 150, m), unit="D"))
                .strftime("%Y-%m-%d"),

    "product": rng.choice(
        ["Airtime", "Bill Pay", "Send Money", "Merchant Pay"],
        m,
        p=[0.3, 0.2, 0.35, 0.15]
    ),

    "amount": rng.lognormal(6.8, 0.9, m).round(0),
})

experiment = pd.DataFrame({
    "user_id": range(1, 3001),
    "group": rng.choice(["A", "B"], 3000)
})

is_b = experiment["group"] == "B"

experiment["converted"] = np.where(
    is_b,
    rng.random(3000) < 0.14,
    rng.random(3000) < 0.11
).astype(int)

users.to_csv("users.csv", index=False)
txns.to_csv("transactions.csv", index=False)
experiment.to_csv("experiment.csv", index=False)

Overwriting generate_data.py


In [2]:
!python generate_data.py

In [3]:
import os

files = [
    "generate_data.py",
    "users.csv",
    "transactions.csv",
    "experiment.csv"
]

for file in files:
    print(file, "->", os.path.exists(file))

generate_data.py -> True
users.csv -> True
transactions.csv -> True
experiment.csv -> True


In [4]:
import pandas as pd

users = pd.read_csv("users.csv")
transactions = pd.read_csv("transactions.csv")
experiment = pd.read_csv("experiment.csv")

print("Users:", users.shape)
print("Transactions:", transactions.shape)
print("Experiment:", experiment.shape)

Users: (3000, 4)
Transactions: (12000, 5)
Experiment: (3000, 3)


## Database Setup

The three generated CSV files are loaded into a SQLite database named `pesapap.db`.

- `users.user_id` is the primary key for the users table.
- `transactions.txn_id` is the primary key for the transactions table.
- `transactions.user_id` is a foreign key referencing `users.user_id`.
- `experiment.user_id` is the primary key for the experiment table and also references `users.user_id`.

In [5]:
import sqlite3
import pandas as pd
import numpy as np
from scipy import stats

users = pd.read_csv("users.csv")
transactions = pd.read_csv("transactions.csv")
experiment = pd.read_csv("experiment.csv")

conn = sqlite3.connect("pesapap.db")

users.to_sql("users", conn, if_exists="replace", index=False)
transactions.to_sql("transactions", conn, if_exists="replace", index=False)
experiment.to_sql("experiment", conn, if_exists="replace", index=False)

print(pd.read_sql("SELECT COUNT(*) AS users FROM users;", conn))
print(pd.read_sql("SELECT COUNT(*) AS transactions FROM transactions;", conn))
print(pd.read_sql("SELECT COUNT(*) AS experiment_rows FROM experiment;", conn))

   users
0   3000
   transactions
0         12000
   experiment_rows
0             3000


## Data Validation

Before analysing growth, we check whether any transactions occurred before the corresponding user's signup date. Such rows would be logically inconsistent because a user should not transact before creating an account.

In [6]:
validation_query = """
SELECT COUNT(*) AS invalid_transactions
FROM transactions t
JOIN users u
    ON t.user_id = u.user_id
WHERE date(t.txn_date) < date(u.signup_date);
"""

validation_result = pd.read_sql(validation_query, conn)

validation_result

,invalid_transactions
0,3737


### Validation Interpretation

Transactions that occurred before the user's signup date are treated as invalid for analyses that depend on the user's lifecycle, such as RFM and cohort retention. They should be excluded from those customer-behaviour calculations because they violate the expected sequence of signup followed by transaction. The raw data will remain unchanged for transparency.

## SQL Analysis

### Question A: Total Transaction Value and Count by Product

This query shows the total transaction value and number of transactions for each product.

In [7]:
query_a = """
SELECT
    product,
    COUNT(*) AS transaction_count,
    SUM(amount) AS total_value
FROM transactions
GROUP BY product
ORDER BY total_value DESC;
"""

result_a = pd.read_sql(query_a, conn)

result_a

,product,transaction_count,total_value
0,Send Money,4161,5639308.0
1,Airtime,3662,4708104.0
2,Bill Pay,2388,3224392.0
3,Merchant Pay,1789,2407505.0


### Question B: Top Five Counties by Transaction Value

This query joins transactions to users and ranks counties by total transaction value.

In [8]:
query_b = """
SELECT
    u.county,
    SUM(t.amount) AS total_value
FROM transactions t
JOIN users u
    ON t.user_id = u.user_id
GROUP BY u.county
ORDER BY total_value DESC
LIMIT 5;
"""

result_b = pd.read_sql(query_b, conn)

result_b

,county,total_value
0,Mombasa,3464271.0
1,Nairobi,3403364.0
2,Nakuru,3138079.0
3,Kisumu,3059055.0
4,Eldoret,2914540.0


### Question C: Monthly Transaction Value and Running Total

This query groups transactions by month using `strftime` and uses a window function to calculate the cumulative transaction value over time.

In [9]:
query_c = """
SELECT
    strftime('%Y-%m', txn_date) AS month,
    SUM(amount) AS monthly_value,
    SUM(SUM(amount)) OVER (
        ORDER BY strftime('%Y-%m', txn_date)
    ) AS running_total
FROM transactions
GROUP BY month
ORDER BY month;
"""

result_c = pd.read_sql(query_c, conn)

result_c

,month,monthly_value,running_total
0,2026-01,1733121.0,1733121.0
1,2026-02,2993105.0,4726226.0
2,2026-03,3199125.0,7925351.0
3,2026-04,3262447.0,11187798.0
4,2026-05,3365002.0,14552800.0
5,2026-06,1426509.0,15979309.0


### Question D: Transaction Value by Acquisition Channel

This query joins transactions to users and calculates the total transaction value generated by each acquisition channel.

In [10]:
query_d = """
SELECT
    u.acquisition_channel,
    SUM(t.amount) AS total_value
FROM transactions t
JOIN users u
    ON t.user_id = u.user_id
GROUP BY u.acquisition_channel
ORDER BY total_value DESC;
"""

result_d = pd.read_sql(query_d, conn)

result_d

,acquisition_channel,total_value
0,Search,4279089.0
1,Agent,3954250.0
2,Social,3926296.0
3,Referral,3819674.0


### Question E: Top Ten Users by Transaction Value

This query calculates each user's total transaction value and uses `RANK()` to identify the highest-value users.

In [11]:
query_e = """
WITH user_value AS (
    SELECT
        user_id,
        SUM(amount) AS total_value
    FROM transactions
    GROUP BY user_id
)
SELECT
    user_id,
    total_value,
    RANK() OVER (ORDER BY total_value DESC) AS value_rank
FROM user_value
ORDER BY value_rank
LIMIT 10;
"""

result_e = pd.read_sql(query_e, conn)

result_e

,user_id,total_value,value_rank
0,1962,29244.0,1
1,2643,29144.0,2
2,2894,28008.0,3
3,2671,27317.0,4
4,50,27158.0,5
5,2279,26634.0,6
6,2837,25643.0,7
7,514,24747.0,8
8,670,23041.0,9
9,2681,22850.0,10


### Question F: Products with Average Transaction Above KES 1,300

This query calculates the average transaction amount for each product and uses `HAVING` to keep only products whose average transaction exceeds KES 1,300.

In [12]:
query_f = """
SELECT
    product,
    AVG(amount) AS average_transaction
FROM transactions
GROUP BY product
HAVING AVG(amount) > 1300
ORDER BY average_transaction DESC;
"""

result_f = pd.read_sql(query_f, conn)

result_f

,product,average_transaction
0,Send Money,1355.277097
1,Bill Pay,1350.247906
2,Merchant Pay,1345.726663


### Question G: Number of Users in Each Value Tier

This query first calculates total transaction value per user in a CTE, then uses `CASE WHEN` to classify users as High, Medium or Low value.

- High: total value ≥ KES 10,000
- Medium: total value ≥ KES 4,000
- Low: total value below KES 4,000

In [13]:
query_g = """
WITH user_value AS (
    SELECT
        user_id,
        SUM(amount) AS total_value
    FROM transactions
    GROUP BY user_id
),
tiered_users AS (
    SELECT
        user_id,
        total_value,
        CASE
            WHEN total_value >= 10000 THEN 'High'
            WHEN total_value >= 4000 THEN 'Medium'
            ELSE 'Low'
        END AS value_tier
    FROM user_value
)
SELECT
    value_tier,
    COUNT(*) AS user_count
FROM tiered_users
GROUP BY value_tier
ORDER BY user_count DESC;
"""

result_g = pd.read_sql(query_g, conn)

result_g

,value_tier,user_count
0,Medium,1317
1,Low,1281
2,High,345


### Question H: Month-over-Month Change in Active Users

This query counts distinct active users in each month, uses `LAG()` to retrieve the previous month's active users, and calculates both the numeric change and whether activity increased, decreased or stayed the same.

In [14]:
query_h = """
WITH monthly_active AS (
    SELECT
        strftime('%Y-%m', txn_date) AS month,
        COUNT(DISTINCT user_id) AS active_users
    FROM transactions
    GROUP BY month
),
with_previous AS (
    SELECT
        month,
        active_users,
        LAG(active_users) OVER (ORDER BY month) AS previous_month_users
    FROM monthly_active
)
SELECT
    month,
    active_users,
    previous_month_users,
    active_users - previous_month_users AS mom_change,
    CASE
        WHEN previous_month_users IS NULL THEN 'No previous month'
        WHEN active_users > previous_month_users THEN 'Increase'
        WHEN active_users < previous_month_users THEN 'Decrease'
        ELSE 'No change'
    END AS change_direction
FROM with_previous
ORDER BY month;
"""

result_h = pd.read_sql(query_h, conn)

result_h

,month,active_users,previous_month_users,mom_change,change_direction
0,2026-01,1055,NaN,NaN,No previous month
1,2026-02,1596,1055.0,541.0,Increase
2,2026-03,1649,1596.0,53.0,Increase
3,2026-04,1704,1649.0,55.0,Increase
4,2026-05,1690,1704.0,-14.0,Decrease
5,2026-06,907,1690.0,-783.0,Decrease


## KPI Scorecard

The KPI scorecard summarises PesaPap's core transaction and user activity metrics. Each KPI is paired with a clear business definition so that the metric can be interpreted consistently.

In [15]:
# Prepare transaction dates
transactions["txn_date"] = pd.to_datetime(transactions["txn_date"])

# Core KPIs
total_value = transactions["amount"].sum()
transaction_count = transactions["txn_id"].nunique()
active_users = transactions["user_id"].nunique()
average_transaction_value = total_value / transaction_count
avg_txns_per_active_user = transaction_count / active_users

monthly_value = (
    transactions
    .set_index("txn_date")
    .resample("MS")["amount"]
    .sum()
)

mom_growth = (monthly_value.pct_change() * 100).round(1)

print(f"Total transaction value: KES {total_value:,.0f}")
print(f"Number of transactions: {transaction_count:,}")
print(f"Active users: {active_users:,}")
print(f"Average transaction value: KES {average_transaction_value:,.2f}")
print(f"Average transactions per active user: {avg_txns_per_active_user:.2f}")

print("\nMonth-over-month growth:")
print(mom_growth)

Total transaction value: KES 15,979,309
Number of transactions: 12,000
Active users: 2,943
Average transaction value: KES 1,331.61
Average transactions per active user: 4.08

Month-over-month growth:
txn_date
2026-01-01     NaN
2026-02-01    72.7
2026-03-01     6.9
2026-04-01     2.0
2026-05-01     3.1
2026-06-01   -57.6
Freq: MS, Name: amount, dtype: float64


In [16]:
metric_dictionary = pd.DataFrame({
    "Metric": [
        "Total Transaction Value",
        "Number of Transactions",
        "Active Users",
        "Average Transaction Value",
        "Average Transactions per Active User",
        "Month-over-Month Growth"
    ],
    "Definition": [
        "Sum of all transaction amounts during the analysis period.",
        "Count of unique transaction IDs during the analysis period.",
        "Count of distinct users who completed at least one transaction.",
        "Total transaction value divided by number of transactions.",
        "Number of transactions divided by number of active users.",
        "Percentage change in total transaction value compared with the previous month."
    ],
    "Calculation": [
        "SUM(amount)",
        "COUNT(DISTINCT txn_id)",
        "COUNT(DISTINCT user_id)",
        "SUM(amount) / COUNT(DISTINCT txn_id)",
        "COUNT(DISTINCT txn_id) / COUNT(DISTINCT user_id)",
        "(Current month value - Previous month value) / Previous month value × 100"
    ]
})

metric_dictionary

,Metric,Definition,Calculation
0,Total Transaction Value,Sum of all transaction amounts during the anal...,SUM(amount)
1,Number of Transactions,Count of unique transaction IDs during the ana...,COUNT(DISTINCT txn_id)
2,Active Users,Count of distinct users who completed at least...,COUNT(DISTINCT user_id)
3,Average Transaction Value,Total transaction value divided by number of t...,SUM(amount) / COUNT(DISTINCT txn_id)
4,Average Transactions per Active User,Number of transactions divided by number of ac...,COUNT(DISTINCT txn_id) / COUNT(DISTINCT user_id)
5,Month-over-Month Growth,Percentage change in total transaction value c...,(Current month value - Previous month value) /...


## Customer Analytics

### RFM Segmentation

RFM analysis scores customers using three behaviours:

- Recency: how recently the user made a transaction.
- Frequency: how many transactions the user made.
- Monetary: how much total value the user transacted.

Higher RFM scores represent stronger customer value.

In [17]:
# Use only lifecycle-valid transactions
users["signup_date"] = pd.to_datetime(users["signup_date"])
transactions["txn_date"] = pd.to_datetime(transactions["txn_date"])

valid_tx = transactions.merge(
    users[["user_id", "signup_date"]],
    on="user_id",
    how="left"
)

valid_tx = valid_tx[
    valid_tx["txn_date"] >= valid_tx["signup_date"]
].copy()

snapshot = valid_tx["txn_date"].max() + pd.Timedelta(days=1)

rfm = valid_tx.groupby("user_id").agg(
    recency=("txn_date", lambda d: (snapshot - d.max()).days),
    frequency=("txn_id", "count"),
    monetary=("amount", "sum")
)

rfm["R"] = pd.qcut(
    rfm["recency"],
    4,
    labels=[4, 3, 2, 1]
).astype(int)

freq_rank = rfm["frequency"].rank(method="first")

rfm["F"] = pd.qcut(
    freq_rank,
    4,
    labels=[1, 2, 3, 4]
).astype(int)

rfm["M"] = pd.qcut(
    rfm["monetary"],
    4,
    labels=[1, 2, 3, 4]
).astype(int)

rfm["score"] = rfm["R"] + rfm["F"] + rfm["M"]

rfm["segment"] = pd.cut(
    rfm["score"],
    bins=[2, 5, 8, 10, 12],
    labels=["At risk", "Needs attention", "Loyal", "Champions"]
)

rfm.head()

,recency,frequency,monetary,R,F,M,score,segment
user_id,,,,,,,,
1,18,3,2418.0,3,2,2,7,Needs attention
2,2,3,2010.0,4,2,2,8,Needs attention
3,25,3,2066.0,2,2,2,6,Needs attention
4,15,6,5746.0,3,4,4,11,Champions
5,26,2,1945.0,2,1,2,5,At risk


In [18]:
rfm_summary = (
    rfm.groupby("segment", observed=True)
    .agg(
        users=("monetary", "count"),
        total_value=("monetary", "sum"),
        average_value=("monetary", "mean")
    )
)

rfm_summary["value_share_pct"] = (
    rfm_summary["total_value"]
    / rfm_summary["total_value"].sum()
    * 100
).round(1)

rfm_summary.round(2)

,users,total_value,average_value,value_share_pct
segment,,,,
At risk,674,845119.0,1253.89,7.7
Needs attention,991,3087541.0,3115.58,28.2
Loyal,641,3697918.0,5768.98,33.8
Champions,410,3326200.0,8112.68,30.4


### Cohort Retention

Cohort retention groups users by the month of their first valid transaction and tracks the percentage who remain active in later months. The most recent cohort cells may be incomplete because the dataset ends before those cohorts have had enough time to reach later months.

In [19]:
cohort_tx = valid_tx.copy()

cohort_tx["txn_month"] = cohort_tx["txn_date"].dt.to_period("M")

cohort_tx["cohort"] = (
    cohort_tx.groupby("user_id")["txn_month"]
    .transform("min")
)

cohort_tx["months_since"] = (
    cohort_tx["txn_month"] - cohort_tx["cohort"]
).apply(lambda x: x.n)

cohort_counts = (
    cohort_tx
    .groupby(["cohort", "months_since"])["user_id"]
    .nunique()
    .unstack(fill_value=0)
)

retention = (
    cohort_counts
    .div(cohort_counts[0], axis=0)
    * 100
)

retention.round(1)

months_since,0,1,2,3,4,5
cohort,,,,,,
2026-01,100.0,52.4,56.0,51.8,58.1,33.0
2026-02,100.0,57.5,56.6,54.3,30.7,0.0
2026-03,100.0,57.5,57.8,29.0,0.0,0.0
2026-04,100.0,56.9,28.8,0.0,0.0,0.0
2026-05,100.0,34.6,0.0,0.0,0.0,0.0
2026-06,100.0,0.0,0.0,0.0,0.0,0.0


In [20]:
max_month = cohort_tx["txn_month"].max()

incomplete_mask = pd.DataFrame(
    False,
    index=retention.index,
    columns=retention.columns
)

for cohort in retention.index:
    for month_num in retention.columns:
        expected_month = cohort + month_num
        if expected_month > max_month:
            incomplete_mask.loc[cohort, month_num] = True

retention_flagged = retention.round(1).astype(object)

for cohort in retention_flagged.index:
    for month_num in retention_flagged.columns:
        if incomplete_mask.loc[cohort, month_num]:
            retention_flagged.loc[cohort, month_num] = "Incomplete"

retention_flagged

months_since,0,1,2,3,4,5
cohort,,,,,,
2026-01,100.0,52.4,56.0,51.8,58.1,33.0
2026-02,100.0,57.5,56.6,54.3,30.7,Incomplete
2026-03,100.0,57.5,57.8,29.0,Incomplete,Incomplete
2026-04,100.0,56.9,28.8,Incomplete,Incomplete,Incomplete
2026-05,100.0,34.6,Incomplete,Incomplete,Incomplete,Incomplete
2026-06,100.0,Incomplete,Incomplete,Incomplete,Incomplete,Incomplete


### Customer Analytics Interpretation

Marketing should target the **At risk** segment first because these users have weaker recent engagement and may be more likely to stop using PesaPap without a re-engagement effort. A win-back campaign, reminder or targeted incentive could help reactivate them.

The cohort retention table should be read from left to right for each cohort. Month 0 is always 100% because it represents the month users first became active. Later percentages show how many users returned in following months. Cells marked **Incomplete** should not be interpreted as poor retention because the dataset has not yet observed those cohorts for that many months.

## Onboarding Experiment

### Hypotheses

**H₀:** The simplified onboarding screen in group B does not change the conversion rate compared with group A.

**H₁:** The simplified onboarding screen in group B changes the conversion rate compared with group A.

The experiment uses conversion rate as the outcome. We first check group balance, then compare conversion rates, calculate relative lift, run a chi-square test, and report a 95% confidence interval for the difference.

In [21]:
# Group balance
group_balance = experiment["group"].value_counts().sort_index()

print("Group sizes:")
print(group_balance)

# Conversion summary
conversion_summary = (
    experiment.groupby("group")
    .agg(
        users=("user_id", "count"),
        conversions=("converted", "sum"),
        conversion_rate=("converted", "mean")
    )
)

conversion_summary["conversion_rate_pct"] = (
    conversion_summary["conversion_rate"] * 100
).round(2)

conversion_summary

Group sizes:
group
A    1443
B    1557
Name: count, dtype: int64


,users,conversions,conversion_rate,conversion_rate_pct
group,,,,
A,1443,151,0.104643,10.46
B,1557,215,0.138086,13.81


In [22]:
# Conversion rates
n_a = conversion_summary.loc["A", "users"]
n_b = conversion_summary.loc["B", "users"]

conv_a = conversion_summary.loc["A", "conversions"]
conv_b = conversion_summary.loc["B", "conversions"]

p_a = conv_a / n_a
p_b = conv_b / n_b

relative_lift = (p_b / p_a - 1) * 100

print(f"Conversion A: {p_a:.2%}")
print(f"Conversion B: {p_b:.2%}")
print(f"Relative lift: {relative_lift:.2f}%")

Conversion A: 10.46%
Conversion B: 13.81%
Relative lift: 31.96%


In [23]:
# Chi-square test
contingency = np.array([
    [conv_a, n_a - conv_a],
    [conv_b, n_b - conv_b]
])

chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

# 95% confidence interval for absolute difference
diff = p_b - p_a

se = np.sqrt(
    p_a * (1 - p_a) / n_a
    + p_b * (1 - p_b) / n_b
)

z = stats.norm.ppf(0.975)

ci_low = diff - z * se
ci_high = diff + z * se

print(f"Chi-square statistic: {chi2:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Absolute lift: {diff:.2%}")
print(f"95% CI: {ci_low:.2%} to {ci_high:.2%}")

Chi-square statistic: 7.5106
P-value: 0.0061
Absolute lift: 3.34%
95% CI: 1.01% to 5.67%


In [24]:
import numpy as np
from scipy import stats

### Experiment Interpretation

The chi-square test produced a p-value of **0.0061**, which is below the 0.05 significance level. Therefore, we reject H₀ and conclude that the conversion rate differs significantly between the two onboarding variants.

Group B produced a **3.34 percentage-point absolute lift** in conversion compared with group A. The 95% confidence interval for the difference is **1.01 to 5.67 percentage points**. Because the entire interval is above zero, the result supports a real improvement rather than random variation.

The simplified onboarding screen should be considered the winning variant. However, the final rollout decision should also consider the business cost of implementation, any effect on downstream user quality, and whether the improvement remains stable after launch.

In [25]:
print(f"Group A conversion rate: {p_a:.2%}")
print(f"Group B conversion rate: {p_b:.2%}")
print(f"Relative lift: {relative_lift:.2f}%")

Group A conversion rate: 10.46%
Group B conversion rate: 13.81%
Relative lift: 31.96%


### Conversion Results

Group A had a conversion rate of **10.46%**, while group B had a conversion rate of **13.81%**.

The simplified onboarding screen therefore produced a **31.96% relative lift** in conversion compared with the current onboarding flow.

The absolute lift was **3.34 percentage points**, and the 95% confidence interval ranged from **1.01 to 5.67 percentage points**. The chi-square test returned **p = 0.0061**, so the difference is statistically significant at the 5% level.

## Estimated Business Impact

To estimate the impact of rolling out the simplified onboarding screen, assume PesaPap receives **10,000 new sign-ups per month**.

Group A converted at **10.46%**, while group B converted at **13.81%**, an absolute lift of **3.34 percentage points**.

At 10,000 monthly sign-ups:

- Expected conversions with group A: about **1,046**
- Expected conversions with group B: about **1,381**
- Estimated additional conversions: about **334 per month**

This suggests that rolling out the simplified onboarding screen could generate roughly **334 additional converted users each month**, assuming the experiment effect remains stable after launch.

In [26]:
import os

os.makedirs("sql", exist_ok=True)

In [27]:
%%writefile sql/queries.sql

-- Validation: How many transactions occurred before the user's signup date?
SELECT COUNT(*) AS invalid_transactions
FROM transactions t
JOIN users u
    ON t.user_id = u.user_id
WHERE date(t.txn_date) < date(u.signup_date);


-- Question A: What are the total transaction value and transaction count by product?
SELECT
    product,
    COUNT(*) AS transaction_count,
    SUM(amount) AS total_value
FROM transactions
GROUP BY product
ORDER BY total_value DESC;


-- Question B: Which five counties generate the highest total transaction value?
SELECT
    u.county,
    SUM(t.amount) AS total_value
FROM transactions t
JOIN users u
    ON t.user_id = u.user_id
GROUP BY u.county
ORDER BY total_value DESC
LIMIT 5;


-- Question C: What is monthly transaction value and the cumulative running total?
SELECT
    strftime('%Y-%m', txn_date) AS month,
    SUM(amount) AS monthly_value,
    SUM(SUM(amount)) OVER (
        ORDER BY strftime('%Y-%m', txn_date)
    ) AS running_total
FROM transactions
GROUP BY month
ORDER BY month;


-- Question D: How much transaction value is generated by each acquisition channel?
SELECT
    u.acquisition_channel,
    SUM(t.amount) AS total_value
FROM transactions t
JOIN users u
    ON t.user_id = u.user_id
GROUP BY u.acquisition_channel
ORDER BY total_value DESC;


-- Question E: Who are the top ten users by total transaction value?
WITH user_value AS (
    SELECT
        user_id,
        SUM(amount) AS total_value
    FROM transactions
    GROUP BY user_id
)
SELECT
    user_id,
    total_value,
    RANK() OVER (ORDER BY total_value DESC) AS value_rank
FROM user_value
ORDER BY value_rank
LIMIT 10;


-- Question F: Which products have an average transaction value above KES 1,300?
SELECT
    product,
    AVG(amount) AS average_transaction
FROM transactions
GROUP BY product
HAVING AVG(amount) > 1300
ORDER BY average_transaction DESC;


-- Question G: How many users fall into High, Medium and Low total-value tiers?
WITH user_value AS (
    SELECT
        user_id,
        SUM(amount) AS total_value
    FROM transactions
    GROUP BY user_id
),
tiered_users AS (
    SELECT
        user_id,
        total_value,
        CASE
            WHEN total_value >= 10000 THEN 'High'
            WHEN total_value >= 4000 THEN 'Medium'
            ELSE 'Low'
        END AS value_tier
    FROM user_value
)
SELECT
    value_tier,
    COUNT(*) AS user_count
FROM tiered_users
GROUP BY value_tier
ORDER BY user_count DESC;


-- Question H: How did active users change month over month?
WITH monthly_active AS (
    SELECT
        strftime('%Y-%m', txn_date) AS month,
        COUNT(DISTINCT user_id) AS active_users
    FROM transactions
    GROUP BY month
),
with_previous AS (
    SELECT
        month,
        active_users,
        LAG(active_users) OVER (ORDER BY month) AS previous_month_users
    FROM monthly_active
)
SELECT
    month,
    active_users,
    previous_month_users,
    active_users - previous_month_users AS mom_change,
    CASE
        WHEN previous_month_users IS NULL THEN 'No previous month'
        WHEN active_users > previous_month_users THEN 'Increase'
        WHEN active_users < previous_month_users THEN 'Decrease'
        ELSE 'No change'
    END AS change_direction
FROM with_previous
ORDER BY month;

Overwriting sql/queries.sql


In [28]:
import os

os.makedirs("reports", exist_ok=True)

In [29]:
%%writefile reports/growth_memo.md
# PesaPap Growth Memo

## Result

The simplified onboarding screen in group B outperformed the current onboarding flow. Group A converted at 10.46%, while group B converted at 13.81%, producing a 31.96% relative lift and a 3.34 percentage-point absolute increase. The chi-square test returned p = 0.0061, which is below the 0.05 significance level. The 95% confidence interval for the absolute lift ranges from 1.01 to 5.67 percentage points. Because the full interval is above zero, the result supports a real improvement in conversion rather than random variation.

## Impact

Assuming PesaPap receives 10,000 new sign-ups per month, the current onboarding flow would produce about 1,046 conversions, while the simplified flow would produce about 1,381. This represents roughly 334 additional converted users per month if the experiment effect remains stable after rollout.

## Recommendation

Roll out the simplified onboarding screen to all eligible new users. Continue monitoring conversion after launch to confirm that the improvement holds outside the experiment. Marketing should also prioritise the At risk RFM segment with targeted win-back activity while protecting high-value Champions.

## Caveats

The business-impact estimate depends on the assumption of 10,000 monthly sign-ups and assumes the experiment lift continues after rollout. The analysis does not measure the cost of implementation or whether converted users remain active long term. Retention results should also exclude incomplete cohort periods, and transactions that occurred before signup should not be used in lifecycle-based customer analysis.

Overwriting reports/growth_memo.md


In [30]:
%%writefile .gitignore
*.db
users.csv
transactions.csv
experiment.csv

Overwriting .gitignore


In [31]:
%%writefile requirements.txt
pandas
numpy
scipy

Overwriting requirements.txt


In [32]:
%%writefile README.md
# PesaPap Growth Analytics

## Project Summary

This project analyses PesaPap's transaction growth, customer behaviour and onboarding experiment using SQL, pandas and inferential statistics. The analysis answers key growth questions, builds a KPI scorecard, segments customers with RFM analysis, measures cohort retention and evaluates whether the simplified onboarding screen should be rolled out.

## Database Tables

### users
- Primary key: `user_id`
- Contains each user's signup date, county and acquisition channel.

### transactions
- Primary key: `txn_id`
- Foreign key: `user_id` references `users.user_id`
- Contains transaction date, product and transaction amount.

### experiment
- Primary key: `user_id`
- Foreign key: `user_id` references `users.user_id`
- Contains experiment group assignment and conversion outcome.

## How to Run

1. Run `generate_data.py` to create `users.csv`, `transactions.csv` and `experiment.csv`.
2. Open `pesapap_analysis.ipynb`.
3. Restart the kernel and run all cells from top to bottom.
4. The notebook builds `pesapap.db`, runs all SQL queries, calculates KPIs, performs RFM and cohort analysis, and evaluates the onboarding experiment.
5. SQL queries are also stored in `sql/queries.sql`.
6. The final growth recommendation is available in `reports/growth_memo.md`.

## Headline Recommendation

Roll out the simplified onboarding screen. Group B achieved a 13.81% conversion rate compared with 10.46% for group A, representing a 31.96% relative lift and a 3.34 percentage-point absolute increase. The result was statistically significant with p = 0.0061, and the 95% confidence interval for the lift was 1.01 to 5.67 percentage points. Assuming 10,000 monthly sign-ups, the new flow could generate roughly 334 additional converted users per month. After rollout, PesaPap should continue monitoring conversion and retention while targeting At risk customers with re-engagement campaigns.

Overwriting README.md


### Group Balance Check

The experiment groups are similar in size, so there is no obvious allocation imbalance between groups A and B. This supports a fair comparison of their conversion rates.